# Mean–variance optimization — companion notebook

Runnable companion for [`mean-variance.md`](./mean-variance.md).

1. Build $\mu, \Sigma$ for 5 synthetic assets.
2. Compute and plot the efficient frontier in $(\sigma, r)$ space.
3. Mark the global minimum-variance and tangency portfolios; draw the capital market line.
4. Estimation-error sensitivity: tiny perturbations of $\mu$, huge weight swings.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

In [ ]:
# 5 assets: expected annual returns and an SPD covariance matrix
mu = np.array([0.06, 0.08, 0.10, 0.12, 0.15])
vols = np.array([0.10, 0.14, 0.18, 0.22, 0.30])

# Random correlation matrix via A A^T construction
A = rng.standard_normal((5, 5))
C = A @ A.T
d = np.sqrt(np.diag(C))
corr = C / np.outer(d, d)
Sigma = corr * np.outer(vols, vols)

print('mu     =', mu)
print('vols   =', vols)
print('corr   =\n', corr)
print('Sigma eigenvalues =', np.linalg.eigvalsh(Sigma))

In [ ]:
ones = np.ones_like(mu)
Sinv = np.linalg.inv(Sigma)
A_ = ones @ Sinv @ ones
B_ = ones @ Sinv @ mu
C_ = mu @ Sinv @ mu
D_ = A_ * C_ - B_**2

def frontier_weights(r_target):
    lam = (A_ * r_target - B_) / D_
    gam = (C_ - B_ * r_target) / D_
    return Sinv @ (lam * mu + gam * ones)

def frontier_vol(r_target):
    return np.sqrt((A_ * r_target**2 - 2 * B_ * r_target + C_) / D_)

# Global minimum-variance portfolio
w_mv = Sinv @ ones / A_
r_mv = w_mv @ mu
s_mv = np.sqrt(w_mv @ Sigma @ w_mv)
print(f'Min-variance weights = {w_mv}')
print(f'  return = {r_mv:.4f}  vol = {s_mv:.4f}')

# Tangency portfolio for a given risk-free rate
rf = 0.03
w_tan_raw = Sinv @ (mu - rf * ones)
w_tan = w_tan_raw / w_tan_raw.sum()
r_tan = w_tan @ mu
s_tan = np.sqrt(w_tan @ Sigma @ w_tan)
sharpe = (r_tan - rf) / s_tan
print(f'\nTangency weights = {w_tan}')
print(f'  return = {r_tan:.4f}  vol = {s_tan:.4f}  Sharpe = {sharpe:.4f}')

In [ ]:
r_grid = np.linspace(mu.min() - 0.02, mu.max() + 0.05, 200)
s_grid = frontier_vol(r_grid)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(s_grid, r_grid, 'tab:blue', lw=2, label='efficient frontier (risky only)')

# Individual assets
ax.scatter(vols, mu, color='black', zorder=5)
for i, v in enumerate(vols):
    ax.annotate(f'A{i+1}', (v, mu[i]), xytext=(5, 5), textcoords='offset points')

# Special portfolios
ax.plot(s_mv, r_mv, 'o', color='tab:green', ms=10, label='min-variance')
ax.plot(s_tan, r_tan, '*', color='tab:red', ms=16, label='tangency')
ax.plot(0, rf, 's', color='tab:orange', ms=8, label=f'risk-free (rf={rf:.2f})')

# Capital market line
s_line = np.linspace(0, s_grid.max() * 1.05, 50)
ax.plot(s_line, rf + sharpe * s_line, '--', color='tab:red', lw=1.5, label='capital market line')

ax.set_xlabel('portfolio volatility'); ax.set_ylabel('expected return')
ax.set_title('Efficient frontier and capital market line')
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_xlim(0, s_grid.max() * 1.1)
plt.show()

## Estimation-error sensitivity

Perturb $\mu$ by tiny iid Normal noise (1% of vol) and re-solve. The tangency portfolio weights swing dramatically.

In [ ]:
n_trials = 200
noise_scale = 0.01 * vols  # 1% of each asset's vol
weights = np.zeros((n_trials, len(mu)))
for i in range(n_trials):
    mu_pert = mu + rng.standard_normal(len(mu)) * noise_scale
    w_raw = Sinv @ (mu_pert - rf * ones)
    weights[i] = w_raw / w_raw.sum()

print('Base tangency weights:        ', w_tan)
print('Perturbed mean weights:       ', weights.mean(axis=0))
print('Perturbed weight std-dev:     ', weights.std(axis=0))
print('Perturbed weight (min, max):  ')
for i in range(len(mu)):
    print(f'  A{i+1}: [{weights[:,i].min():+.3f}, {weights[:,i].max():+.3f}]')

fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot([weights[:, i] for i in range(len(mu))],
           labels=[f'A{i+1}' for i in range(len(mu))])
ax.scatter(range(1, len(mu) + 1), w_tan, color='tab:red', zorder=5, label='base tangency')
ax.axhline(0, color='gray', lw=0.5)
ax.set_ylabel('weight')
ax.set_title('Tangency weights under 1%-of-vol perturbations to mu')
ax.legend(); plt.show()